# PART 1 — PySpark Fundamentals & Environment
### A complete beginner-to-expert PySpark learning 


## 1. What is Spark? Spark Architecture

### What is Apache Spark?
Apache Spark is a **distributed, in-memory data processing engine**. Instead of processing
a huge file on ONE machine (slow, may not even fit in memory), Spark **splits the data into
chunks (partitions) and processes those chunks in parallel across many machines** (a cluster).

**Real-life analogy:**
Imagine you must count all books in a 10-floor library by yourself — it would take hours.
Now imagine you hire 10 people, one per floor, all counting **at the same time**, and then
you add up their totals. That's exactly what Spark does with data: it divides the work
(partitions) and does it in parallel (distributed computing).

### Spark Architecture — 3 Main Components

```
                        ┌────────────────────────────┐
                        │        CLUSTER MANAGER      │
                        │  (YARN / Kubernetes / Databricks) │
                        │  Decides which machines      │
                        │  (nodes) are available        │
                        └───────────────┬──────────────┘
                                         │ allocates resources
                                         ▼
  ┌───────────────────────────────────────────────────────────────────┐
  │                              DRIVER                                │
  │  - Runs your main() program / notebook code                        │
  │  - Creates the SparkSession                                        │
  │  - Builds the DAG (Directed Acyclic Graph) of operations           │
  │  - Splits work into JOBS -> STAGES -> TASKS, sends to executors     │
  │  - Collects final results back                                     │
  └───────────────┬───────────────────────┬───────────────┬────────────┘
                  │                       │               │
                  ▼                       ▼               ▼
         ┌─────────────────┐   ┌─────────────────┐  ┌─────────────────┐
         │   EXECUTOR 1    │   │   EXECUTOR 2    │  │   EXECUTOR 3    │
         │  runs TASKS     │   │  runs TASKS     │  │  runs TASKS     │
         │  holds data in  │   │  holds data in  │  │  holds data in  │
         │  memory/disk    │   │  memory/disk    │  │  memory/disk    │
         └─────────────────┘   └─────────────────┘  └─────────────────┘
             (Worker Node 1)       (Worker Node 2)      (Worker Node 3)
```

| Component | Role | Real-life analogy |
|---|---|---|
| **Driver** | The "brain" — plans the work, breaks it into tasks, coordinates everything | Restaurant Manager who takes the order and assigns dishes to chefs |
| **Executors** | Worker processes that actually do the computation and hold data in memory/disk | Chefs, each cooking their assigned dish |
| **Cluster Manager** | Allocates machines/resources (CPU, RAM) to the Driver & Executors | Restaurant Owner who decides how many chefs are hired today |

### Job → Stage → Task hierarchy (how the Driver breaks down work)
```
  ACTION (e.g. .show(), .count())
       │
       ▼
     JOB  (one job is triggered per ACTION)
       │
       ├── STAGE 1  (a group of transformations that don't need a SHUFFLE)
       │      ├── Task 1  (runs on partition 1)
       │      ├── Task 2  (runs on partition 2)
       │      └── Task 3  (runs on partition 3)
       │
       └── STAGE 2  (new stage starts whenever data must be SHUFFLED, e.g. groupBy/join)
              ├── Task 1
              └── Task 2
```
**Key idea — Transformations vs Actions:**
- **Transformations** (select, filter, withColumn...) are **LAZY** — Spark just builds a plan, nothing runs yet.
- **Actions** (show, count, collect, write...) **TRIGGER** actual execution across the cluster.
This laziness lets Spark's Catalyst Optimizer look at your ENTIRE chain of transformations and find the fastest execution plan before running anything — like a GPS calculating the best route before you start driving, instead of turn-by-turn with no overview.

In [0]:
# In Databricks, a SparkSession called `spark` is ALREADY created for you automatically.
# You do NOT need to create it yourself in a Databricks notebook (unlike plain PySpark scripts).

# Let's inspect the Driver and cluster info to see the architecture in action.
print("Spark Version:", spark.version)                       # Shows the Spark engine version running on the Driver
print("App Name:", spark.sparkContext.appName)                # Name of this Spark Application
print("Master:", spark.sparkContext.master)                   # Shows cluster manager connection info
print("Default Parallelism (approx total cores across executors):",
      spark.sparkContext.defaultParallelism)                  # How many tasks can run in parallel = total cores available

In [0]:
# ---- See Jobs/Stages/Tasks live: run a transformation chain then trigger an action ----
sample_df = spark.range(1, 1000001)                    # LAZY: just builds a plan for numbers 1 to 1,000,000 (nothing executes yet)
sample_df2 = sample_df.withColumn("double_val", sample_df.id * 2)   # LAZY: still just a plan
sample_df3 = sample_df2.filter(sample_df2.double_val > 500000)       # LAZY: still just a plan

print("Nothing has run yet - transformations are lazy!")

result_count = sample_df3.count()                       # ACTION: THIS triggers a real JOB on the cluster
print("Count of rows where double_val > 500000:", result_count)
# TIP: Open the "Spark UI" tab (top of notebook results) -> Jobs -> you'll see this single .count() created 1 Job with Stages/Tasks

In [0]:
sample_df3.show()

In [0]:
print(sample_df3)

In [0]:
sample_df3.display()

In [0]:
display(sample_df3.limit(100))

##RDD

### Resilient distributed dataset

1. they are slow then df and they do not know the schema


In [0]:
data=[("Amit", 5000), ("Ravi", 6000), ("Ravi", 7000)]

rdd=spark.sparkContext.parallelize(data)

rdd_result=rdd.filter(lambda x:x[1]>48000)
print("Rdd result",rdd_result.collet() )



#Dataframe


Data frame if like multidimension array, they are faster then rdd


In [0]:
data=[{"name":"Amit", "salary":50000}, {"name":"Ravi", "salary":60000}, {"name":"Ravi", "salary":70000}]

df=spark.createDataFrame(data)

# df.filter(df.salary>50000).show()  # if u want to display in single line 

newdf= df.filter(df.salary>50000).select("name")
display(newdf)

# Word count programm

In [0]:
word_rdd=spark.sparkContext.parallelize([
    "Pyspark is a spark API for python",
    "Spark is a distributed computing system",
    "pyspark is python api for spark"

])


#map function in rdd

upper_rdd=word_rdd.map(lambda line:line.upper())
print(upper_rdd.collect())

#Output= PYSPARK..... 


#flatmap() #divide the elemts into multiple elements  
split_words=word_rdd.flatMap(lambda line:line.split(" "))
print(split_words.collect())


#output:  ["Pyspark", "is", "a", "spark", "API", "for", "python", "Spark", "is", "a", "distributed", "computing", "system", "pyspark", "is", "python", "api", is a spark API for python]


# reduce function in rdd  : reduceByKey() 

word_pairs=split_words.map((lambda word:(word,1)))
print(word_pairs.collect())

#output: [('Pyspark', 1), ('is', 1), ('a', 1), ('spark', 1), ('API', 1), ('for', 1), ('python', 1), ('Spark', 1), ('is', 1), ('a', 1), ('distributed', 1), ('computing', 1), ('system', 1), ('pyspark', 1), ('is', 1)))


word_count=word_pairs.reduceByKey(lambda a,b:a+b)
print(word_count.collect())

#output: [('Pyspark', 2), ('a', 2), ('spark', 2), ('API', 1), ('for', 1), ('python', 1), ('distributed', 1), ('computing', 1), ('is', 3




### Word count using Dataframe

In [0]:
from pyspark.sql.functions import upper, col, split, count, explode

data=[
     ("Pyspark is a spark API for python"),
   ("Spark is a distributed computing system"),
    ("pyspark is python api for spark")

]

df=spark.createDataFrame(data, ["sentence"])


# use upper fun df
upper_df=df.select(
    upper(col("sentence")).alias("uppersentence")
)

# split the elements into multiple words

slipt_words=df.select(
    explode(
        split(col("sentence"), " ")
        ).alias("word")

    )

# recude by key  to count the number of words

word_count=(
    slipt_words
        .groupBy("word")
        .agg(count("*").alias("count"))
)


# word_count.display()

# Show words that appear exactly 2 times
word_count.filter(col("word") == "api").show()


To see how many active spark apps are running

In [0]:
# from pyspark.sql import SparkSession
# active_spark=SparkSession.getActiveSession()

# print("Active session app", active_spark.sparkContext.appName)


DF from list of tuples


In [0]:
employee = [
    (1, "Amit", "IT", 50000, 89),
    (2, "Ravi", "IT", 60000, 90),
    (3, "Neha", "HR", 45000, 88),
    (4, "Priya", "Finance", 70000, 92),
    (5, "Rahul", "Sales", 55000, 85),
    (6, "Ankit", "IT", 65000, 91),
    (7, "Pooja", "Marketing", 58000, 87),
    (8, "Vikas", "Finance", 75000, 94),
    (9, "Sneha", "HR", 47000, 89),
    (10, "Karan", "Sales", 62000, 90),
    (11, "Rohit", "IT", 72000, 93),
    (12, "Anjali", "Marketing", 53000, 86),
    (13, "Deepak", "Finance", 81000, 95),
    (14, "Nisha", "HR", 49000, 90),
    (15, "Suresh", "Sales", 57000, 84),
    (16, "Meena", "IT", 83000, 96),
    (17, "Arjun", "Marketing", 61000, 88),
    (18, "Kavita", "Finance", 78000, 93),
    (19, "Ajay", "HR", 51000, 87),
    (20, "Ritu", "Sales", 59000, 89),
    (21, "Mohit", "IT", 90000, 97),
    (22, "Divya", "Marketing", 64000, 91),
    (23, "Sanjay", "Finance", 85000, 94),
    (24, "Komal", "HR", 52000, 88),
    (25, "Manish", "Sales", 61000, 90),
    (26, "Nitin", "IT", 95000, 98),
    (27, "Shreya", "Marketing", 66000, 92),
    (28, "Vivek", "Finance", 88000, 95),
    (29, "Payal", "HR", 54000, 89),
    (30, "Aakash", "Sales", 63000, 91),
    (31, "Harsh", "IT", 99000, 96),
    (32, "Simran", "Marketing", 68000, 93),
    (33, "Yash", "Finance", 91000, 97),
    (34, "Rohan", "HR", 56000, 90),
    (35, "Tanya", "Sales", 65000, 92),
    (36, "Gaurav", "IT", 105000, 98),
    (37, "Isha", "Marketing", 70000, 94),
    (38, "Abhishek", "Finance", 93000, 96),
    (39, "Kirti", "HR", 58000, 91),
    (40, "Varun", "Sales", 67000, 93),
    (41, "Sahil", "IT", 108000, 99),
    (42, "Riya", "Marketing", 72000, 95),
    (43, "Naveen", "Finance", 96000, 97),
    (44, "Preeti", "HR", 60000, 92),
    (45, "Tarun", "Sales", 69000, 94),
    (46, "Asha", "IT", 112000, 98),
    (47, "Lokesh", "Marketing", 74000, 96),
    (48, "Bhavna", "Finance", 98000, 99),
    (49, "Vinay", "HR", 62000, 93),
    (50, "Sonia", "Sales", 71000, 95)
]

columns = ["emp_id", "emp_name", "department", "salary", "rating"]

df=spark.createDataFrame(employee, columns)

display(df)



DF from list of  dict 

In [0]:
employyes=[
    {"id":1,"name":"Amit", "department":"IT", "salary":50000},
    {"id":2,"name":"Ravi", "department":"IT", "salary":60000},
    {"id":3,"name":"Neha", "department":"HR", "salary":45000},
    
]

dffromdict=spark.createDataFrame(employyes)
display(dffromdict)


DF from RDD

In [0]:
employyes=[
    {"id":1,"name":"Amit", "department":"IT", "salary":50000},
    {"id":2,"name":"Ravi", "department":"IT", "salary":60000},
    {"id":3,"name":"Neha", "department":"HR", "salary":45000},
    
]

columns=["id","name","department","salary"]
rdd=spark.sparkContext.parallelize(employyes)
dffromrdd=rdd.toDf(columns)
display(dffromrdd)



DF from Range

In [0]:
df_range=spark.range(1, 101)
df_range.display()

DF from pandas

In [0]:
import pandas as pd

pandas_df=pd.DataFrame(
    
    {
        "id":[1,2,3],
        "name":["Amit","Shena","Raghu"],
        "salary":[5000,40000,6000]

}
    )


df_from_pandas=spark.createDataFrame(pandas_df)
# df_from_pandas.display()
df_from_pandas.printSchema()

# covert DF to Pandas

pandas_12=df_from_pandas.toPandas()
print(type(pandas_12))



Data frame with schema

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

schema_for_dataframe=StructType(
    [
        StructField("Id", IntegerType(), True),
        StructField("Name", StringType(), True),

    ]
)

df=spark.createDataFrame([],schema_for_dataframe)
# df.display()

df.printSchema()

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType


employee_schema=StructType([
    StructField("id", IntegerType(), False), #id should not be null
    StructField("name", StringType(), True),
    StructField("department", StringType(), True),
    StructField("salary", DoubleType(), True),
    StructField("age", IntegerType(), True)
   
])

employeeData=[
    (1, "Amit", "IT", 50000.0, 68),
    (2, "Ravi", "IT", 60000.0, 50),
    (3, "Neha", "HR", 45000.0, 45),
    (4, "Raghu", "HR", 40000.0, 35),
    (5, "Priya", "IT", 55000.0, 40),
    (6, "Rajesh", "IT", 65000.0,45)
]

df=spark.createDataFrame(employeeData, employee_schema)

df.printSchema()
df.display()

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

employee_schema=StructType([
    StructField("id", IntegerType(), False), #id should not be null
    StructField("name", StringType(), True),


   StructField("address", StructType(
       [
        StructField("street", StringType(), True),
        StructField("city", StringType(), True),
        
   ]))
   
])

nested_data=[
    (1, "Amit", {"street": "123 Main St", "city": "New York"}),
    (2, "Ravi", {"street": "456 Oak Ave", "city": "Los Angeles"})
]

df_nested=spark.createDataFrame(nested_data, schema=employee_schema)
df_nested.printSchema()

df_nested.select("name", "address.street", "address.city").display() 



In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, ArrayType, MapType

array_schema=StructType([
    StructField("id", IntegerType(), True),
    StructField("Skills", ArrayType(StringType()), True),#
    StructField("ratings", MapType(StringType(), IntegerType()), True)#{key, value}

])                              #[all the lements inside array will be having string data type ]

data=[
    (1, ["Python", "Java", "Scala"], {"2026":4, "2025":5}),
    (2, ["SQL", "Python", "C++"], {"2026":5, "2025":4}),
    (3, ["Java", "Scala", "C++"], {"2026":3, "2025":4})
    
]
  
df=spark.createDataFrame(data, schema=array_schema)
df.printSchema()
# df.display()

df.display()

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType


employee_schema=StructType([
    StructField("id", IntegerType(), False), #id should not be null
    StructField("name", StringType(), True),
    StructField("department", StringType(), True),
    StructField("salary", DoubleType(), True),
    StructField("age", IntegerType(), True)
   
])

employeeData=[
    (1, "Amit", "IT", 50000.0, 68),
    (2, "Ravi", "IT", 60000.0, 50),
    (3, "Neha", "HR", 45000.0, 45),
    (4, "Raghu", "HR", 40000.0, 35),
    (5, "Priya", "IT", 55000.0, 40),
    (6, "Rajesh", "IT", 65000.0,45)
]

df=spark.createDataFrame(employeeData, employee_schema)

df.printSchema()
df.display()

Describe the df

In [0]:
df.describe().show()

In [0]:
df.show()# will give u only top 20 rows
df.show(2)
df.show(truncate=False)

# to vertically print each row
df.show(2, vertical=True)



In [0]:
df.describe("salary", "age").show()

Summary

In [0]:
df.summary().show()

In [0]:
df.summary("count","mean").show()

Basic functions in pyspark

In [0]:
print("Row count", df.count())
print("Coulumn count", len(df.columns))
print("Columns names", df.columns)
print("Data types", df.dtypes)

# Unique values columns
df.select("department").distinct().show()

print("Check df is emplty:"   , df.isEmpty())

df.select("age","department").show()





# select staements

In [0]:
df.select("age","department").show()
df.select(df.age, df.department).show()
df.select(df["age"], df["department"]).show()
df.select("*").show()

df.select([c for c in df.columns if c!="age"]).show()

## filter commands

In [0]:
# df.filter(df.salary>60000).show()
# df.where(df.department=="IT").show()
# df.filter(df.department != "IT").show()  # & = and | = or
# df.filter((df.department=="IT") & (df.salary>50000)).show()
# df.filter((df.department=="IT") | (df.salary>50000)).show()

# df.filter(~(df.department == "HR")).show() 

# df.filter(df.name.isin("Amit","Ravi" )).show()

# df.filter(df.name.like("R%")).show()

# df.filter(df.salary.between(50000,60000)).show()

# df.filter(df.age.isNull()).show()

# df.filter(df.age.isNotNull()).show()


df.filter("salary >50000 AND department = 'IT' ").show()



### Creating columns in DF

In [0]:
from pyspark.sql.functions import col, upper, lower

df.withColumn("Upper_name", upper(col("name"))).show()

df.withColumn("bonus", col("salary")*0.10).show()

In [0]:
df_multi=(
    df
    .withColumn("bonus", col("salary")*0.10)
    .withColumn("Upper_name", upper(col("name")))
    .withColumn("Lower_name", lower(col("name")))

)
df_multi.display()


Catalog-> Schema->Tables or Volumes

In [0]:
df=spark.read.json("/Volumes/pysparkcatalog/pyspark_schema/pyspark_volume")

In [0]:
df = spark.read.option("multiLine", "true").json("/Volumes/pysparkcatalog/pyspark_schema/pyspark_volume")
display(df)

In [0]:
display(df)